# Cache Lifecycle — TTL, Expiry, and Invalidation

A semantic cache that never expires can serve stale answers.
Medha 0.3.1 provides full control over the cache lifecycle:

- **TTL per entry** — `store(..., ttl=N)` sets an expiry N seconds from now
- **Default TTL** — `Settings(default_ttl_seconds=N)` applies to all stores
- **Manual cleanup** — `expire()` removes all expired entries on demand
- **Automatic cleanup** — `Settings(cleanup_interval_seconds=N)` starts a background task
- **Targeted invalidation** — by question, query hash, template, or entire collection

**When to use TTL:**
- Schema changes invalidate queries automatically after N seconds
- Seasonal data (daily stats, live prices) should expire frequently
- Bug-fix releases: short TTL ensures stale cached queries age out quickly

**Requirements:** `pip install "medha-archai[fastembed]"`

In [ ]:
import asyncio

from medha import Medha, Settings, SearchStrategy
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

## Cell 1: TTL per Entry — `store(..., ttl=10)` + Sleep + Search → NO_MATCH

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("lifecycle_demo", embedder=embedder, settings=settings) as medha:
    await medha.store(
        "Current stock price of ACME",
        "SELECT price FROM stocks WHERE ticker = 'ACME'",
        ttl=2,  # expires in 2 seconds
    )
    await medha.store(
        "Total users",
        "SELECT COUNT(*) FROM users",
        ttl=3600,  # expires in 1 hour
    )

    hit_before = await medha.search("ACME stock price")
    print(f"Before expiry : strategy={hit_before.strategy.value}")

    await asyncio.sleep(3)  # wait for the 2s entry to expire

    hit_after = await medha.search("ACME stock price")
    print(f"After expiry  : strategy={hit_after.strategy.value}")
    assert hit_after.strategy == SearchStrategy.NO_MATCH, "Entry should have expired"

    hit_long = await medha.search("How many users are there?")
    print(f"Long-lived    : strategy={hit_long.strategy.value}  (still valid)")

## Cell 2: Default TTL — `Settings(default_ttl_seconds=3600)`

When `default_ttl_seconds` is set, every `store()` call without an explicit
`ttl` argument inherits this default. Store-time `ttl=...` overrides the default.

In [ ]:
settings = Settings(
    backend_type="memory",
    default_ttl_seconds=3600,  # all entries expire in 1 hour unless overridden
)

async with Medha("default_ttl", embedder=embedder, settings=settings) as medha:
    await medha.store("User count", "SELECT COUNT(*) FROM users")
    # Explicit ttl overrides the default
    await medha.store("Live price", "SELECT price FROM tickers WHERE id = 1", ttl=60)

    # Inspect entries via scroll to see expires_at
    results, _ = await medha._backend.scroll("default_ttl", limit=10)
    for r in results:
        print(f"  {r.original_question!r:35s}  expires_at={r.expires_at}")

## Cell 3: `ttl=None` — Force Immortality Despite Default TTL

Passing `ttl=None` explicitly overrides `default_ttl_seconds` and stores
the entry without an expiry — it never expires.

In [ ]:
settings = Settings(
    backend_type="memory",
    default_ttl_seconds=60,  # default: expire in 60s
)

async with Medha("immortal", embedder=embedder, settings=settings) as medha:
    await medha.store(
        "Schema version",
        "SELECT version FROM schema_migrations ORDER BY id DESC LIMIT 1",
        ttl=None,  # explicitly immortal — ignores default_ttl_seconds
    )

    results, _ = await medha._backend.scroll("immortal", limit=5)
    for r in results:
        print(f"  {r.original_question!r:30s}  expires_at={r.expires_at}  (None = immortal)")

## Cell 4: Manual Cleanup — `await medha.expire()`

`expire()` scans the backend for entries whose `expires_at` is in the past
and deletes them. Returns the number of entries removed.

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("expire_demo", embedder=embedder, settings=settings) as medha:
    await medha.store("Entry A", "SELECT 'A'", ttl=1)
    await medha.store("Entry B", "SELECT 'B'", ttl=1)
    await medha.store("Entry C", "SELECT 'C'", ttl=3600)

    count_before = await medha._backend.count("expire_demo")
    print(f"Count before expire() : {count_before}")

    await asyncio.sleep(2)

    n = await medha.expire()
    print(f"Removed {n} expired entries")

    count_after = await medha._backend.count("expire_demo")
    print(f"Count after expire()  : {count_after}")

## Cell 5: Automatic Cleanup — `cleanup_interval_seconds`

When `cleanup_interval_seconds` is set, `Medha.start()` launches a background
asyncio task that calls `expire()` at the specified interval. No manual
intervention required.

In [ ]:
settings = Settings(
    backend_type="memory",
    cleanup_interval_seconds=60,  # background task runs every 60 seconds
)

print(f"cleanup_interval_seconds : {settings.cleanup_interval_seconds}")
print("On medha.start(), a background asyncio.Task will call expire() every 60s.")
print("On medha.close(), the background task is cancelled automatically.")

# Demonstrate that the task starts and stops correctly
async with Medha("auto_cleanup", embedder=embedder, settings=settings) as medha:
    await medha.store("Some query", "SELECT 1", ttl=1)
    print(f"\nBackground cleanup task running: {medha._cleanup_task is not None}")

## Cell 6: `invalidate(question)` → `True` / `False`

Invalidates a specific entry by its exact question string. Returns `True` if
the entry was found and removed, `False` if not found.

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("inv_demo", embedder=embedder, settings=settings) as medha:
    await medha.store("List all admins", "SELECT * FROM users WHERE role = 'admin'")
    await medha.store("User count",      "SELECT COUNT(*) FROM users")

    removed = await medha.invalidate("List all admins")
    print(f"invalidate('List all admins') → {removed}")

    removed_again = await medha.invalidate("List all admins")
    print(f"invalidate again (already gone) → {removed_again}")

    hit = await medha.search("Show all admin users")
    print(f"Search after invalidation: {hit.strategy.value}")

## Cell 7: `invalidate_by_query_hash(hash)` → count

In [ ]:
import hashlib

settings = Settings(backend_type="memory")

async with Medha("hash_inv", embedder=embedder, settings=settings) as medha:
    await medha.store("Count users",       "SELECT COUNT(*) FROM users")
    await medha.store("How many users?",   "SELECT COUNT(*) FROM users")  # same SQL → same hash
    await medha.store("Total user count",  "SELECT COUNT(*) FROM users")  # same SQL → same hash

    # Compute the query hash (SHA-256 of the normalized SQL)
    sql = "SELECT COUNT(*) FROM users"
    query_hash = hashlib.sha256(sql.strip().lower().encode()).hexdigest()

    n = await medha.invalidate_by_query_hash(query_hash)
    print(f"invalidate_by_query_hash(SHA256 of SQL) removed {n} entries")

## Cell 8: `invalidate_by_template(template_id)` → count

In [ ]:
from medha.types import QueryTemplate

settings = Settings(backend_type="memory")

async with Medha("tmpl_inv", embedder=embedder, settings=settings) as medha:
    template = QueryTemplate(
        intent="users_by_status",
        template_text="Show {status} users",
        query_template="SELECT * FROM users WHERE status = '{status}'",
        parameters=["status"],
    )
    await medha.load_templates([template])

    # Store entries tagged with the template_id
    await medha.store("Active users",  "SELECT * FROM users WHERE status = 'active'",  template_id="users_by_status")
    await medha.store("Blocked users", "SELECT * FROM users WHERE status = 'blocked'", template_id="users_by_status")
    await medha.store("User count",    "SELECT COUNT(*) FROM users")  # different template

    n = await medha.invalidate_by_template("users_by_status")
    print(f"invalidate_by_template('users_by_status') removed {n} entries")

    count = await medha._backend.count("tmpl_inv")
    print(f"Remaining entries: {count}")

## Cell 9: `invalidate_collection()` → count

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("coll_inv", embedder=embedder, settings=settings) as medha:
    for i in range(5):
        await medha.store(f"Question {i}", f"SELECT {i}")

    count_before = await medha._backend.count("coll_inv")
    print(f"Count before: {count_before}")

    n = await medha.invalidate_collection()
    print(f"invalidate_collection() removed {n} entries")

    count_after = await medha._backend.count("coll_inv")
    print(f"Count after : {count_after}")

## Pattern: Short TTL for Expensive or Volatile Queries

The most common lifecycle pattern is **short TTL for volatile data + no TTL
for stable schema queries**:

```python
# Stable schema query — cache forever
await medha.store("List all table names",
                  "SELECT table_name FROM information_schema.tables",
                  ttl=None)

# Live metric — expire every 5 minutes
await medha.store("Current DAU",
                  "SELECT COUNT(DISTINCT user_id) FROM events WHERE DATE(ts) = CURDATE()",
                  ttl=300)

# Price feed — expire every 30 seconds
await medha.store("ACME stock price",
                  "SELECT price FROM tickers WHERE symbol = 'ACME'",
                  ttl=30)
```

Pair with `cleanup_interval_seconds` to keep the backend lean automatically:
```python
Settings(default_ttl_seconds=3600, cleanup_interval_seconds=300)
```